# Denoising Method Comparison — POC_DDM

Compares **three** denoising methods on raw kinetic (LAMP/DDM) curves, each applied with
its current chosen hyperparameter (no HP sweep re-run in this notebook):

| # | Method | Hyperparameter |
|---|---|---|
| 1 | Moving avg (`ori_curves_avg`) | `config.WINDOW_SIZE_ORI` |
| 2 | Wavelet (universal threshold) | `WAVELETS[0]` = `'sym8'`, `LEVEL`, `THRESH_MODE` |
| 3 | Savitzky-Golay | `SG_POLYORDER`, `SG_OPTIMAL_W` (fixed below) |


In [1]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import joblib
import pywt
from scipy.signal import savgol_filter
from scipy.ndimage import uniform_filter1d

sys.path.insert(0, '/vol/bitbucket/gk225/POC_DDM/gk_code')
sys.path.insert(0, '/vol/bitbucket/gk225/POC_DDM/gk_code/main')
import config

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

print(f'pywt {pywt.__version__} | base: {config.BASE_FOLDER}')


[*] SUCCESS: TensorFlow is utilizing the GPU -> /physical_device:GPU:0

pywt 1.8.0 | base: /vol/bitbucket/gk225/POC_DDM_datasets


In [2]:
# ── Configuration ─────────────────────────────────────────────
# DATASET      = 'POC_DDM_final_nc_subtract'   # or 'POC_DDM_multi'
DATASET      = 'POC_DDM_final'
EXP_FOLDER   = os.path.join(config.BASE_FOLDER, DATASET)
CURVE_TYPE   = 'ori_curves'
WAVELETS     = ['sym8']          # first entry used in comparison
LEVEL        = None
THRESH_MODE  = 'soft'
SG_POLYORDER = 2                 # 2 = quadratic, 3 = cubic
SG_OPTIMAL_W = 69                # current chosen window -- derivative-test sweep on
                                  # D20260807_E00_C00_F4500KHz_U_DDM_02_07

In [3]:
# ── Wavelet ────────────────────────────────────────────────────────────────
def denoise_curve(curve, wavelet, level=LEVEL, mode=THRESH_MODE):
    coeffs     = pywt.wavedec(curve, wavelet, level=level)
    sigma      = np.median(np.abs(coeffs[-1])) / 0.6745
    threshold  = sigma * np.sqrt(2 * np.log(len(curve)))
    new_coeffs = [coeffs[0]] + [pywt.threshold(d, threshold, mode=mode) for d in coeffs[1:]]
    return pywt.waverec(new_coeffs, wavelet)[:len(curve)]

def denoise_all(curves, wavelet):
    it = tqdm(curves, desc=f'Denoising [{wavelet}]', unit='curve') if tqdm else curves
    return np.array([denoise_curve(c, wavelet) for c in it])

# ── SG ─────────────────────────────────────────────────────────────────────────────
def _ensure_odd(w): return w + (1 - w % 2)

def apply_sg(curves, window_length, polyorder):
    w = _ensure_odd(int(window_length)); w = max(w, polyorder + 2)
    return savgol_filter(curves, window_length=w, polyorder=polyorder, axis=1)

# ── Derivative-test sweep -- drives the SG/Wavelet HP search below ─────────────────
def _derivative_scores(curves, param_values, denoise_fn, sample_size=400, seed=0):
    rng    = np.random.default_rng(seed)
    sample = curves[rng.choice(len(curves), size=min(sample_size, len(curves)), replace=False)]
    _ref_w = max(5, _ensure_odd(int(sample.shape[1] * 0.03)))
    ref    = savgol_filter(sample, window_length=_ref_w, polyorder=2, axis=1)
    peak_r = np.abs(np.diff(ref, axis=1)).max(axis=1)
    ro_raw = np.std(np.diff(np.diff(sample, axis=1), axis=1), axis=1)
    prs, ros = [], []
    for p in param_values:
        df = np.diff(denoise_fn(sample, p), axis=1)
        prs.append(np.mean(np.abs(df).max(axis=1) / np.where(peak_r > 0, peak_r, 1)))
        ros.append(np.mean(np.std(np.diff(df, axis=1), axis=1) / np.where(ro_raw > 0, ro_raw, 1)))
    return np.array(prs), np.array(ros)

def _sweet_spot(param_values, roughnesses):
    """First (smallest) param where roughness drops within 2x of its floor (10th pctile)."""
    floor = np.percentile(roughnesses, 10)
    sweet = np.where(roughnesses <= 2.0 * floor)[0]
    return float(param_values[sweet[0]] if len(sweet) else param_values[np.argmin(roughnesses)])

# ── Data loading ────────────────────────────────────────────────────────────────
def list_folders(exp_folder):
    out = []
    for name in sorted(os.listdir(exp_folder)):
        p = os.path.join(exp_folder, name)
        if (os.path.isdir(p) and name not in config.EXCLUDED_FOLDERS
                and os.path.exists(os.path.join(p, config.TRAINING_DATA_PATH))):
            out.append((name, p))
    return out

def load_exp(folder_path):
    d           = joblib.load(os.path.join(folder_path, config.TRAINING_DATA_PATH))
    curves      = np.array(d['curves'][CURVE_TYPE])
    if 'ori_curves_avg' in d['curves']:
        curves_avg = np.array(d['curves']['ori_curves_avg'])
    else:
        curves_avg = uniform_filter1d(np.array(d['curves']['ori_curves']), size=config.WINDOW_SIZE_ORI, axis=1, mode='nearest')
    well_labels = np.array(d['well_labels'])
    return curves, curves_avg, well_labels

print('All functions loaded.')

All functions loaded.


In [ ]:
import re
import pandas as pd
from scipy.stats import pearsonr

# ── Colours & display constants ───────────────────────────────────────────────
METHOD_COLORS  = ['#CC79A7', '#E69F00', '#0072B2']
SG_POLY_COLORS = {2: '#0072B2', 3: '#009E73', 4: '#D55E00'}
_M_COLOR       = {'raw': '#999999', 'smoothed': '#CC79A7', 'sg': '#0072B2'}
_WAVELET_COLORS_CYCLE = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#882255']

# HP search candidates (feed the SG/Wavelet HP search cells below)
SG_POLYORDERS      = [2, 3, 4]
WAVELET_CANDIDATES = [
    # 'db4', 'db6', 'db8',
    'sym4', 'sym6', 'sym8',
    # 'coif2', 'coif4',
    # 'bior3.5', 'bior4.4',
]

# ── Shared helpers ────────────────────────────────────────────────────────────
def _safe_corr(a, b):
    """Pearson r; returns np.nan if either input is constant."""
    return np.nan if np.std(a) == 0 or np.std(b) == 0 else pearsonr(a, b)[0]


def _full_metrics(raw, denoised):
    """Mean per-sample SNR, noise%, TV ratio, Pearson fidelity."""
    noise  = raw - denoised
    ns     = np.std(noise, axis=1)
    sr     = raw.max(axis=1) - raw.min(axis=1)
    vd, vn = np.var(denoised, axis=1), np.var(noise, axis=1)
    tv_d   = np.abs(np.diff(denoised, axis=1)).sum(axis=1)
    tv_r   = np.abs(np.diff(raw,      axis=1)).sum(axis=1)
    with np.errstate(divide='ignore', invalid='ignore'):
        snr_v   = np.where(vn > 0, 10*np.log10(np.where(vn > 0, vd/vn, 1.0)), np.nan)
        noise_v = np.where(sr > 0, ns / np.where(sr > 0, sr, 1.0) * 100, np.nan)
        tv_v    = tv_d / np.where(tv_r > 0, tv_r, np.nan)
    return {
        'snr':    float(np.nanmean(snr_v)),
        'noise%': float(np.nanmean(noise_v)),
        'tv':     float(np.nanmean(tv_v)),
        'corr':   float(np.nanmean([_safe_corr(raw[i], denoised[i]) for i in range(len(raw))])),
    }


# ── Method resolver ───────────────────────────────────────────────────────────
def _resolve_methods(r, methods):
    """Return [(array, title, color), ...] for the requested method keys.

    Keys:
      'smoothed'       moving average
      'sg'             SG (baseline polyorder + fixed window, see config)
      'sg_p2/3/4'      SG HP search result (per-polyorder auto window)
      'wv_<name>'      wavelet HP candidate  e.g. 'wv_sym6'
      '<name>'         baseline wavelet from WAVELETS  e.g. 'sym8'
    """
    wv_iter = iter(_WAVELET_COLORS_CYCLE)
    out = []
    for key in methods:
        if key == 'raw':
            out.append((r['raw'], 'Raw', _M_COLOR['raw']))
        elif key == 'smoothed':
            out.append((r['smoothed'],
                        f'Smoothed\n(w={config.WINDOW_SIZE_ORI})', _M_COLOR['smoothed']))
        elif key == 'sg':
            out.append((r.get('sg'),
                        f'SG p={SG_POLYORDER}\n(w={SG_OPTIMAL_W})', _M_COLOR['sg']))
        elif key.startswith('sg_p') and key[4:].isdigit():
            entry = r.get(key)
            if entry is not None:
                poly = int(key[4:])
                out.append((entry['denoised'],
                             f'SG p={poly}\n(w={entry["optimal_w"]})',
                             SG_POLY_COLORS.get(poly, '#888888')))
            else:
                print(f'[!] {key!r} not found — run SG HP search first')
        elif key.startswith('wv_'):
            wv_name = key[3:]
            entry   = r.get(key)
            if entry is not None:
                out.append((entry['denoised'],
                             f'Wavelet\n({wv_name})', next(wv_iter, '#888888')))
            elif wv_name in r.get('denoised', {}):
                out.append((r['denoised'][wv_name],
                             f'Wavelet\n({wv_name})', next(wv_iter, '#888888')))
            else:
                print(f'[!] {key!r} not found — run wavelet HP search first')
        elif key in r.get('denoised', {}):
            out.append((r['denoised'][key],
                        f'Wavelet\n({key})', next(wv_iter, '#888888')))
        else:
            print(f'[!] Unknown method key: {key!r}  (skipped)')
    return [(arr, lbl, col) for arr, lbl, col in out if arr is not None]


def _labels():
    return [
        f'Moving avg\n(w={config.WINDOW_SIZE_ORI})',
        f'Wavelet\n({WAVELETS[0]})',
        f'SG p={SG_POLYORDER}\n(w={SG_OPTIMAL_W})',
    ]

def _ready(r): return 'sg' in r


# ── Quantitative comparison ───────────────────────────────────────────────────
def compare_all_methods(folder_name, methods=None, plot=True):
    """
    Compute 7 denoising metrics and (optionally) plot bar charts.

    methods = None  → 3 baseline methods (smoothed, wavelet, sg), current hyperparams.
    methods = list  → any combination via _resolve_methods keys.

    Metrics: SNR, Noise%, Fidelity, AC lag-1, TV ratio, ΔTTP, SD_max ratio.
    """
    r = results[folder_name]
    if not _ready(r): print(f'[!] Run apply cell first for {folder_name}'); return
    raw = r['raw']

    if methods is None:
        labels  = _labels()
        arrays  = [r['smoothed'], r['denoised'][WAVELETS[0]], r['sg']]
        colors  = METHOD_COLORS
    else:
        resolved = _resolve_methods(r, methods)
        labels   = [l for _, l, _ in resolved]
        arrays   = [a for a, _, _ in resolved]
        colors   = [c for _, _, c in resolved]

    def tv(a): return np.abs(np.diff(a, axis=1)).sum(axis=1)
    def ac1(rw, dn):
        res = rw - dn
        return np.nanmean([np.corrcoef((e := res[i]-res[i].mean())[:-1], e[1:])[0, 1]
                           for i in range(len(res)) if res[i].std() > 0])

    mnames = ['SNR (dB)', 'Noise %', 'Fidelity\n(corr)', 'Residual\nAC lag-1',
              'TV ratio', 'Δ TTP', 'SD_max\nratio']
    better = ['↑', '↓', '↑', '↓', '↓', '↓', '→1']

    def best_idx(row, b):
        fin = np.isfinite(row)
        if not fin.any(): return None
        r_ = np.where(fin, row, np.nan)
        if b == '↓': return int(np.nanargmin(r_))
        if b == '↑': return int(np.nanargmax(r_))
        return int(np.nanargmin(np.abs(r_ - 1.0)))

    n_m  = len(labels)
    sc   = np.full((7, n_m), np.nan)
    _ref_w = max(5, _ensure_odd(int(raw.shape[1] * 0.03)))
    _ref   = savgol_filter(raw, window_length=_ref_w, polyorder=2, axis=1)
    draw   = np.abs(np.diff(_ref, axis=1)); ttp_raw = np.argmax(draw, axis=1).astype(float)
    for mi, den in enumerate(arrays):
        noise  = raw - den; ns = np.std(noise, axis=1); sr = raw.max(axis=1) - raw.min(axis=1)
        vd, vn = np.var(den, axis=1), np.var(noise, axis=1)
        dden   = np.abs(np.diff(den, axis=1))
        with np.errstate(divide='ignore', invalid='ignore'):
            sc[0,mi] = np.nanmean(np.where(vn > 0, 10*np.log10(np.where(vn > 0, vd/vn, 1.0)), np.nan))
            sc[1,mi] = np.nanmean(np.where(sr > 0, ns / np.where(sr > 0, sr, 1.0) * 100, np.nan))
        sc[2,mi] = np.nanmean([_safe_corr(raw[i], den[i]) for i in range(len(raw))])
        sc[3,mi] = ac1(raw, den)
        sc[4,mi] = np.nanmean(tv(den) / np.where(tv(raw) > 0, tv(raw), np.nan))
        sc[5,mi] = np.mean(np.abs(ttp_raw - np.argmax(dden, axis=1).astype(float)))
        sc[6,mi] = np.nanmean(dden.max(axis=1) / np.where(draw.max(axis=1)>0, draw.max(axis=1), np.nan))

    all_mn, all_sc, all_bt = mnames, list(sc), better

    if plot:
        fig, axes_pl = plt.subplots(1, len(all_mn), figsize=(max(6, 1.8*n_m), 5))
        if len(all_mn) == 1: axes_pl = [axes_pl]
        fig.suptitle(f'Quantitative Comparison — {folder_name}', fontsize=11, fontweight='bold')
        x = np.arange(n_m)
        for ax, metric, vals, b in zip(axes_pl, all_mn, all_sc, all_bt):
            best = best_idx(vals, b)
            for xi, (val, color) in enumerate(zip(vals, colors)):
                if np.isnan(val):
                    ax.bar(xi, 1, color='none', edgecolor=color, lw=1.5, ls='--', zorder=3)
                    ax.text(xi, 0.5, 'N/A', ha='center', va='center', fontsize=8, color=color, fontweight='bold')
                else:
                    bar = ax.bar(xi, val, color=color, edgecolor='black', zorder=3)
                    if xi == best: bar[0].set_edgecolor('red'); bar[0].set_linewidth(2.5)
                    ax.text(xi, val, f'{val:.3f}', ha='center', va='bottom', fontsize=7.5)
            ax.set_title(metric, fontweight='bold', fontsize=9)
            ax.set_xticks(x)
            ax.set_xticklabels([l.replace('\n',' ') for l in labels], rotation=30, ha='right', fontsize=8)
            ax.grid(axis='y', alpha=0.3, zorder=0)
        axes_pl[0].set_ylabel('↑ higher = better', fontsize=8, color='gray')
        if len(axes_pl) > 3: axes_pl[3].set_ylabel('↓ lower = better', fontsize=8, color='gray')
        if len(axes_pl) > 6: axes_pl[6].set_ylabel('→ 1.0 = best',     fontsize=8, color='gray')
        note = '■ red outline = best  |  N/A = zero noise variance'
        fig.text(0.99, 0.01, note, ha='right', fontsize=7.5, color='red', style='italic')
        fig.tight_layout(); plt.show(); plt.close(fig)

    # Build DataFrame (methods as rows, metrics as columns)
    directions = {mn.replace('\n', ' '): bt for mn, bt in zip(all_mn, all_bt)}
    df = pd.DataFrame(
        {mn.replace('\n', ' '): [float(v) for v in vals]
         for mn, vals in zip(all_mn, all_sc)},
        index=[l.replace('\n', ' ') for l in labels],
    )

    def _style_col(col):
        d      = directions.get(col.name, '↑')
        finite = col.dropna()
        if finite.empty:
            return [''] * len(col)
        best_lbl = ((finite - 1.0).abs().idxmin() if d == '→1'
                    else finite.idxmin()           if d == '↓'
                    else finite.idxmax())
        return ['background-color: #c8f7c5; font-weight: bold'
                if i == best_lbl else '' for i in col.index]

    try:
        from IPython.display import display
        display(df.style.apply(_style_col).format('{:.4f}', na_rep='N/A')
                  .set_caption(folder_name))
    except Exception:
        print(df.round(4).to_string())

    return df


# ── Cross-chip averaging (shared by the "averaged metrics" cell and the LaTeX table) ──
def _method_family_key(label):
    """Groups a method label the same way regardless of chip -- needed because SG HP
    search's per-chip auto-tuned window means the SAME method ('SG p=2') carries a
    DIFFERENT window in its label on every chip ('SG p=2 (w=31)' vs '(w=27)', ...).
    Averaging by the raw label string would treat those as different methods; this
    strips just the w=.. part so they group together. Every other label's
    hyperparameter (Smoothed's w, Wavelet's mother function) is a fixed constant
    across chips already, so it needs no stripping."""
    m = re.match(r'^(SG p=\d+) \(w=\d+\)$', label)
    return m.group(1) if m else label


def _extract_sg_window(label):
    m = re.match(r'^SG p=\d+ \(w=(\d+)\)$', label)
    return int(m.group(1)) if m else None


def _average_metrics_df(metrics_by_folder, folders, metric_cols=None, show_window=True, return_std=False):
    """Method x metric DataFrame averaged across every folder, grouped by
    _method_family_key (not the raw label) so SG HP search's per-chip window doesn't
    split what's really the same method into separate rows. The displayed SG window
    is the mean of each chip's own optimal_w, rounded to the nearest odd integer
    (matching _ensure_odd) and prefixed with '~' since chips didn't all land on the
    same value -- e.g. 'SG p=2 (w=~29)'. Pass show_window=False to drop the
    window annotation entirely (e.g. for a plain method-name display). Pass
    return_std=True to also get the across-chip std, as a second DataFrame with
    the same (grouped, reindexed, relabeled) index -- for reporting spread
    alongside the mean (e.g. 'mean ± std') via the LaTeX average panel.

    metric_cols: which columns to average -- defaults to every column present in the
    per-folder DataFrames (all 7 metrics compare_all_methods computes); pass a subset
    (e.g. _LATEX_METRIC_COLS) to restrict it."""
    used = [metrics_by_folder[f[0]] for f in folders if f[0] in metrics_by_folder]
    if not used:
        raise ValueError('No folders with metrics to average.')
    cols = metric_cols if metric_cols is not None else list(used[0].columns)
    combined = pd.concat(used)
    family_keys = combined.index.map(_method_family_key)
    grouped = combined.groupby(family_keys)[cols]
    avg = grouped.mean()
    std = grouped.std() if return_std else None

    order, seen, sg_windows = [], set(), {}
    for lbl in used[0].index:
        fam = _method_family_key(lbl)
        if fam not in seen:
            seen.add(fam); order.append(fam)
    for df in used:
        for lbl in df.index:
            w = _extract_sg_window(lbl)
            if w is not None:
                sg_windows.setdefault(_method_family_key(lbl), []).append(w)

    def _display_label(fam):
        if fam in sg_windows and show_window:
            return f'{fam} (w=~{_ensure_odd(round(np.mean(sg_windows[fam])))})'
        return fam

    avg = avg.reindex(order)
    avg.index = [_display_label(f) for f in order]
    if return_std:
        std = std.reindex(order)
        std.index = avg.index
        return avg, std
    return avg


# ── Best-value highlighting (mirrors compare_all_methods' own per-column styling) ──
_METRIC_DIRECTIONS = {
    'SNR (dB)': '\u2191', 'Noise %': '\u2193', 'Fidelity (corr)': '\u2191',
    'Residual AC lag-1': '\u2193', 'TV ratio': '\u2193', '\u0394 TTP': '\u2193',
    'SD_max ratio': '\u21921',
}


def _highlight_best(col, directions=_METRIC_DIRECTIONS):
    d = directions.get(col.name, '\u2191')
    finite = col.dropna()
    if finite.empty:
        return [''] * len(col)
    best_lbl = ((finite - 1.0).abs().idxmin() if d == '\u21921'
                else finite.idxmin()           if d == '\u2193'
                else finite.idxmax())
    return ['background-color: #c8f7c5; font-weight: bold'
            if i == best_lbl else '' for i in col.index]

print('All utility functions loaded.')

In [5]:
label_maps = config.get_label_mappings(EXP_FOLDER)
results    = {}
folders    = list_folders(EXP_FOLDER)

print(f'{DATASET}: {len(folders)} folders')
for folder_name, folder_path in folders:
    print(f'\n─── {folder_name} ───')
    try:
        raw, smoothed, well_labels = load_exp(folder_path)
        print(f'  {raw.shape}')
        results[folder_name] = {
            'raw':         raw,
            'smoothed':    smoothed,
            'denoised':    {w: denoise_all(raw, wavelet=w) for w in WAVELETS},
            'well_labels': well_labels,
            'label_map':   label_maps.get(folder_name, {}),
        }
    except Exception as e:
        print(f'  [ERROR] {e}')

POC_DDM_final: 6 folders

─── D20260806_E00_C00_F4500KHz_U_DDM_01_06 ───
  (14507, 813)


Denoising [sym8]: 100%|██████████| 14507/14507 [00:14<00:00, 986.98curve/s] 



─── D20260807_E00_C00_F4500KHz_U_DDM_02_07 ───
  (15420, 827)


Denoising [sym8]: 100%|██████████| 15420/15420 [00:20<00:00, 736.34curve/s] 



─── D20260808_E00_C00_F4500KHz_U_DDM_03_01 ───
  (16526, 403)


...

Denoising [sym8]: 100%|██████████| 16574/16574 [00:14<00:00, 1121.16curve/s]



─── D20260825_E00_C00_F4500KHz_U_DDM_05_01 ───
  (17350, 908)


Denoising [sym8]: 100%|██████████| 17350/17350 [00:17<00:00, 974.30curve/s] 



─── D20260825_E00_C00_F4500KHz_U_DDM_06_02 ───
  (16971, 915)


Denoising [sym8]: 100%|██████████| 16971/16971 [00:22<00:00, 748.96curve/s] 


---
## Savitzky-Golay Smoothing

Fits a polynomial of degree $p$ to each sliding window of $w$ points. Unlike moving
average ($p=1$), it preserves peaks and the S-curve shape. `SG_OPTIMAL_W` above is the
current chosen window (fixed, not re-swept in this notebook).


---
## Hyperparameter Search (per chip)

SG (polyorder × window) and Wavelet (mother function) each use the same derivative-test
sweep (`_derivative_scores` / `_sweet_spot`) to auto-select their window/candidate per
chip — this is what feeds the `sg_p2/p3/p4` and `wv_sym4/sym6/sym8` rows in the table below.


In [6]:
# ── SG HP Search: polyorder × window sweep ───────────────────────────────────
# Saves results[folder]['sg_p{poly}'] — re-running skips cached entries.

for folder_name, _ in folders:
    if folder_name not in results:
        continue
    r   = results[folder_name]
    raw = r['raw']
    T   = raw.shape[1]

    sg_windows = np.unique(np.array(
        [_ensure_odd(int(w)) for w in np.linspace(5, max(7, int(T * 0.25)), 40)]
    ))
    sg_windows = sg_windows[sg_windows >= 5]

    print(f'\n{folder_name}  (T={T})')
    for poly in SG_POLYORDERS:
        key = f'sg_p{poly}'
        if key in r:
            d = r[key]
            print(f'  SG p={poly}: cached  optimal_w={d["optimal_w"]}  '
                  f'SNR={d["metrics"]["snr"]:.1f}dB')
            continue
        fn       = lambda c, w, p=poly: apply_sg(c, w, p)
        prs, ros = _derivative_scores(raw, sg_windows.astype(float), fn)
        opt_w    = int(_sweet_spot(sg_windows.astype(float), ros))
        den      = apply_sg(raw, opt_w, poly)
        m        = _full_metrics(raw, den)
        r[key]   = dict(windows=sg_windows, prs=prs, ros=ros,
                        optimal_w=opt_w, denoised=den, metrics=m)
        print(f'  SG p={poly}: optimal_w={opt_w}  SNR={m["snr"]:.1f}dB  '
              f'TV={m["tv"]:.3f}  corr={m["corr"]:.4f}')

print('\nDone. Keys: sg_p2, sg_p3, sg_p4')


D20260806_E00_C00_F4500KHz_U_DDM_01_06  (T=813)
  SG p=2: optimal_w=101  SNR=13.4dB  TV=0.032  corr=0.9628
  SG p=3: optimal_w=101  SNR=13.4dB  TV=0.032  corr=0.9631
  SG p=4: optimal_w=101  SNR=13.7dB  TV=0.039  corr=0.9652

D20260807_E00_C00_F4500KHz_U_DDM_02_07  (T=827)
  SG p=2: optimal_w=103  SNR=9.6dB  TV=0.030  corr=0.9111
  SG p=3: optimal_w=103  SNR=9.7dB  TV=0.031  corr=0.9118
  SG p=4: optimal_w=103  SNR=10.0dB  TV=0.037  corr=0.9171

D20260808_E00_C00_F4500KHz_U_DDM_03_01  (T=403)
  SG p=2: optimal_w=49  SNR=13.4dB  TV=0.063  corr=0.9564
  SG p=3: optimal_w=49  SNR=13.4dB  TV=0.064  corr=0.9567
  SG p=4: optimal_w=49  SNR=13.7dB  TV=0.075  corr=0.9594

D20260810_E00_C00_F4500KHz_U_DDM_04_01  (T=315)
  SG p=2: optimal_w=39  SNR=8.9dB  TV=0.063  corr=0.8991
  SG p=3: optimal_w=39  SNR=9.0dB  TV=0.064  corr=0.8996
  SG p=4: optimal_w=39  SNR=9.3dB  TV=0.080  corr=0.9072

D20260825_E00_C00_F4500KHz_U_DDM_05_01  (T=908)
  SG p=2: optimal_w=113  SNR=13.9dB  TV=0.028  corr=0.9741

In [7]:
# ── Wavelet HP Search: mother function comparison ─────────────────────────────
# Saves results[folder]['wv_<name>'] — re-running skips cached entries.

for folder_name, _ in folders:
    if folder_name not in results:
        continue
    r   = results[folder_name]
    raw = r['raw']
    T   = raw.shape[1]
    print(f'\n{folder_name}  (T={T})')

    for wv in WAVELET_CANDIDATES:
        key = f'wv_{wv}'
        if key in r:
            print(f'  Wavelet {wv}: cached')
            continue
        if wv in r.get('denoised', {}):
            den    = r['denoised'][wv]
            r[key] = dict(denoised=den, metrics=_full_metrics(raw, den), is_ref=True)
            print(f'  Wavelet {wv}: (already in WAVELETS)  SNR={r[key]["metrics"]["snr"]:.1f}dB')
            continue
        try:
            if pywt.Wavelet(wv).dec_len > T:
                print(f'  Wavelet {wv}: skip (filter > signal length)')
                continue
            den    = denoise_all(raw, wv)
            r[key] = dict(denoised=den, metrics=_full_metrics(raw, den), is_ref=False)
            m      = r[key]['metrics']
            print(f'  Wavelet {wv}: SNR={m["snr"]:.1f}dB  TV={m["tv"]:.3f}  '
                  f'corr={m["corr"]:.4f}')
        except Exception as e:
            print(f'  Wavelet {wv}: ERROR — {e}')

print('\nDone. Keys: wv_<name> for each candidate.')


D20260806_E00_C00_F4500KHz_U_DDM_01_06  (T=813)


Denoising [sym4]: 100%|██████████| 14507/14507 [00:05<00:00, 2520.28curve/s]


  Wavelet sym4: SNR=13.4dB  TV=0.028  corr=0.9624


Denoising [sym6]: 100%|██████████| 14507/14507 [00:05<00:00, 2626.89curve/s]


  Wavelet sym6: SNR=13.4dB  TV=0.028  corr=0.9625
  Wavelet sym8: (already in WAVELETS)  SNR=13.7dB

D20260807_E00_C00_F4500KHz_U_DDM_02_07  (T=827)


...

  Wavelet sym6: SNR=13.9dB  TV=0.025  corr=0.9744
  Wavelet sym8: (already in WAVELETS)  SNR=14.2dB

D20260825_E00_C00_F4500KHz_U_DDM_06_02  (T=915)


Denoising [sym4]: 100%|██████████| 16971/16971 [00:06<00:00, 2756.37curve/s]


  Wavelet sym4: SNR=10.6dB  TV=0.021  corr=0.9363


Denoising [sym6]: 100%|██████████| 16971/16971 [00:05<00:00, 2993.85curve/s]


  Wavelet sym6: SNR=10.9dB  TV=0.023  corr=0.9396
  Wavelet sym8: (already in WAVELETS)  SNR=11.3dB

Done. Keys: wv_<name> for each candidate.


In [8]:
# ── Apply SG to all datasets ──────────────────────────────────────
# Override the current chosen window here if needed:
# SG_OPTIMAL_W = 21

print(f'SG    : w={SG_OPTIMAL_W}, p={SG_POLYORDER}')

for folder_name, _ in folders:
    if folder_name not in results: continue
    raw = results[folder_name]['raw']
    print(f'{folder_name}  ({raw.shape[0]} curves) ...', end=' ', flush=True)
    results[folder_name]['sg'] = apply_sg(raw, SG_OPTIMAL_W, SG_POLYORDER)
    print('done.')

print('\nAll methods applied.')

SG    : w=69, p=2
D20260806_E00_C00_F4500KHz_U_DDM_01_06  (14507 curves) ... done.
D20260807_E00_C00_F4500KHz_U_DDM_02_07  (15420 curves) ... done.
D20260808_E00_C00_F4500KHz_U_DDM_03_01  (16526 curves) ... done.
D20260810_E00_C00_F4500KHz_U_DDM_04_01  (16574 curves) ... done.
D20260825_E00_C00_F4500KHz_U_DDM_05_01  (17350 curves) ... done.
D20260825_E00_C00_F4500KHz_U_DDM_06_02  (16971 curves) ... done.

All methods applied.


In [9]:
denoising_metrics_by_folder = {}
for folder_name, folder_path in folders:
    denoising_metrics = compare_all_methods(
        folder_name, plot=False,
        methods=['smoothed', 'sg_p2', 'sg_p3', 'sg_p4', 'wv_sym4', 'wv_sym6', 'wv_sym8'])
    denoising_metrics_by_folder[folder_name] = denoising_metrics

,SNR (dB),Noise %,Fidelity (corr),Residual AC lag-1,TV ratio,Δ TTP,SD_max ratio
Smoothed (w=50),13.3186,5.4631,0.9626,0.1988,0.0343,184.5838,0.3713
SG p=2 (w=101),13.4079,5.4393,0.9628,0.1904,0.0317,186.9560,0.3405
SG p=3 (w=101),13.4359,5.4224,0.9631,0.1857,0.0325,206.7305,0.3626
SG p=4 (w=101),13.7191,5.2712,0.9652,0.1394,0.0387,224.2531,0.4846
Wavelet (sym4),13.3715,5.4565,0.9624,0.2013,0.0285,194.8792,0.6316
Wavelet (sym6),13.3840,5.4493,0.9625,0.1989,0.0280,195.3649,0.5974
Wavelet (sym8),13.7319,5.2661,0.9651,0.1417,0.0332,198.1645,0.6186


,SNR (dB),Noise %,Fidelity (corr),Residual AC lag-1,TV ratio,Δ TTP,SD_max ratio
Smoothed (w=50),9.5386,6.9420,0.9109,0.1893,0.0330,219.1015,0.3648
SG p=2 (w=103),9.6279,6.9274,0.9111,0.1849,0.0300,224.8196,0.3228
SG p=3 (w=103),9.6727,6.9029,0.9118,0.1794,0.0309,255.6254,0.3621
SG p=4 (w=103),9.9928,6.7125,0.9171,0.1334,0.0372,256.5584,0.4763
Wavelet (sym4),9.5688,6.9550,0.9101,0.1966,0.0264,214.6003,0.6323
Wavelet (sym6),9.5824,6.9480,0.9103,0.1948,0.0260,215.2339,0.6104
Wavelet (sym8),10.0060,6.7045,0.9171,0.1354,0.0317,215.7700,0.6290


,SNR (dB),Noise %,Fidelity (corr),Residual AC lag-1,TV ratio,Δ TTP,SD_max ratio
Smoothed (w=50),12.4955,5.8840,0.9502,0.1477,0.0500,85.9813,0.2851
SG p=2 (w=49),13.3599,5.4830,0.9564,0.0162,0.0628,81.9448,0.3801
SG p=3 (w=49),13.3933,5.4635,0.9567,0.0098,0.0642,85.1799,0.3978
SG p=4 (w=49),13.6899,5.3051,0.9594,-0.0452,0.0755,95.7475,0.4986
Wavelet (sym4),13.1534,5.6040,0.9544,0.0692,0.0530,81.1946,0.5553
Wavelet (sym6),13.1901,5.5890,0.9546,0.0641,0.0521,79.5942,0.5395
Wavelet (sym8),13.6010,5.3571,0.9584,-0.0192,0.0623,80.3729,0.5708


,SNR (dB),Noise %,Fidelity (corr),Residual AC lag-1,TV ratio,Δ TTP,SD_max ratio
Smoothed (w=50),8.0255,7.9457,0.8838,0.1114,0.0441,52.5030,0.1734
SG p=2 (w=39),8.9300,7.4218,0.8991,-0.0356,0.0626,54.8610,0.2764
SG p=3 (w=39),8.9658,7.4046,0.8996,-0.0405,0.0639,62.3343,0.2855
SG p=4 (w=39),9.3419,7.1584,0.9072,-0.0949,0.0802,92.1428,0.3818
Wavelet (sym4),8.6142,7.5970,0.8936,0.0382,0.0483,42.1059,0.5167
Wavelet (sym6),9.1087,7.3051,0.9026,-0.0421,0.0590,46.0089,0.5099
Wavelet (sym8),9.0710,7.3262,0.9019,-0.0392,0.0571,46.5413,0.4417


,SNR (dB),Noise %,Fidelity (corr),Residual AC lag-1,TV ratio,Δ TTP,SD_max ratio
Smoothed (w=50),13.8972,5.2390,0.9744,0.1937,0.0329,159.7548,0.4170
SG p=2 (w=113),13.8758,5.2644,0.9741,0.2005,0.0281,159.0874,0.3600
SG p=3 (w=113),13.9026,5.2486,0.9743,0.1960,0.0288,176.0893,0.3748
SG p=4 (w=113),14.1793,5.0965,0.9757,0.1481,0.0346,199.3688,0.4782
Wavelet (sym4),13.6662,5.3715,0.9731,0.2372,0.0233,182.3522,0.6139
Wavelet (sym6),13.9260,5.2374,0.9744,0.1971,0.0250,181.5893,0.6027
Wavelet (sym8),14.2349,5.0675,0.9760,0.1422,0.0307,179.4965,0.6257


,SNR (dB),Noise %,Fidelity (corr),Residual AC lag-1,TV ratio,Δ TTP,SD_max ratio
Smoothed (w=50),10.8743,6.4493,0.9399,0.1743,0.0314,181.7193,0.4064
SG p=2 (w=113),10.8807,6.4738,0.9394,0.1794,0.0264,181.7451,0.3431
SG p=3 (w=113),10.9153,6.4541,0.9398,0.1747,0.0272,206.1333,0.3679
SG p=4 (w=113),11.1996,6.2831,0.9431,0.1299,0.0332,226.4433,0.4736
Wavelet (sym4),10.6247,6.6226,0.9363,0.2210,0.0211,197.4390,0.6042
Wavelet (sym6),10.9117,6.4576,0.9396,0.1802,0.0227,198.7892,0.5924
Wavelet (sym8),11.2734,6.2421,0.9438,0.1221,0.0292,199.3188,0.6212


### Averaged metrics across all chips

All 7 metrics from `compare_all_methods`, averaged across every chip. SG rows are grouped
by polyorder only (`SG p=2`/`p=3`/`p=4`) -- the per-chip auto-tuned window is stripped
before averaging (`_method_family_key`) since it differs across chips; the displayed window
is the mean of each chip's own value, marked `w=~..`.


In [17]:
avg_denoising_metrics = _average_metrics_df(denoising_metrics_by_folder, folders, show_window=False)
print(f"Averaged across {len(folders)} chips:")
try:
    from IPython.display import display
    display(avg_denoising_metrics.style.apply(_highlight_best).format('{:.4f}', na_rep='N/A'))
except Exception:
    print(avg_denoising_metrics.round(4).to_string())


Averaged across 6 chips:


,SNR (dB),Noise %,Fidelity (corr),Residual AC lag-1,TV ratio,Δ TTP,SD_max ratio
Smoothed (w=50),11.3583,6.3205,0.9370,0.1692,0.0376,147.2739,0.3363
SG p=2,11.6804,6.1683,0.9405,0.1227,0.0403,148.2356,0.3371
SG p=3,11.7143,6.1494,0.9409,0.1175,0.0412,165.3488,0.3585
SG p=4,12.0204,5.9711,0.9446,0.0685,0.0499,182.4190,0.4655
Wavelet (sym4),11.4998,6.2678,0.9383,0.1606,0.0334,152.0952,0.5923
Wavelet (sym6),11.6838,6.1644,0.9407,0.1322,0.0355,152.7634,0.5754
Wavelet (sym8),11.9864,5.9939,0.9437,0.0805,0.0407,153.2773,0.5845


### LaTeX table -- Residual AC lag-1 / SNR (dB) / Fidelity (corr) per chip

One sub-table per chip (`DDM_0x` -> `Chip 0x`), restricted to the three metrics
above. Methods are grouped as `Smoothed: Simple Moving Average` / `SG:
Savitzky-Golay` / `Wavelet: DWT`, with the per-row hyperparameter
(`w=`/`p=.. w=..`/`sym..`) split into its own column.

In [22]:
import re
import string

def _chip_label(folder_name):
    m = re.search(r'DDM_(\d+)', folder_name)
    return f'Chip {int(m.group(1)):02d}' if m else folder_name


def _split_method_label(label):
    """'Smoothed (w=15)' -> ('Smoothed: Simple Moving Average', '(w=15)')
       'SG p=2 (w=31)'   -> ('SG: Savitzky-Golay', '(p=2 w=31)')
       'SG p=2 (w=~29)'  -> ('SG: Savitzky-Golay', '(p=2 w=~29)')  -- averaged-panel window
       'Wavelet (sym4)'  -> ('Wavelet: DWT', '(sym4)')"""
    m = re.match(r'^SG p=(\d+) \(w=(~?\d+)\)$', label)
    if m:
        return 'SG: Savitzky-Golay', f'(p={m.group(1)} w={m.group(2)})'
    m = re.match(r'^Smoothed \((w=\d+)\)$', label)
    if m:
        return 'Smoothed: Simple Moving Average', f'({m.group(1)})'
    m = re.match(r'^Wavelet \((.+)\)$', label)
    if m:
        return 'Wavelet: DWT', f'({m.group(1)})'
    return label, ''


_LATEX_METRIC_COLS      = ['Residual AC lag-1', 'SNR (dB)', 'Fidelity (corr)']
_LATEX_METRIC_HEADERS   = ['Residual Autocorrelation (Lag-1)', 'SNR (dB)', 'Fidelity (Corr)']
_LATEX_METRIC_FMT       = ['{:.3f}', '{:.2f}', '{:.3f}']
_LATEX_METRIC_DIRECTION = ['min', 'max', 'max']  # residual AC: lower is better; SNR/fidelity: higher is better


def _latex_panel(df, panel_letter, chip_name, std_df=None):
    best_row = {
        col: (df[col].idxmin() if direction == 'min' else df[col].idxmax())
        for col, direction in zip(_LATEX_METRIC_COLS, _LATEX_METRIC_DIRECTION)
    }

    lines = [
        f'    ({panel_letter}) Performance on {chip_name}\\\\[0.5em]',
        '    \\resizebox{\\textwidth}{!}{',
        '    \\begin{tabular}{llrrr}',
        '    \\toprule',
        '    \\textbf{Method} & \\textbf{Hyperparameter} & \\textbf{'
        + '} & \\textbf{'.join(_LATEX_METRIC_HEADERS) + '} \\\\',
        '    \\midrule',
    ]
    for method_label, row in df.iterrows():
        method_name, hp = _split_method_label(method_label)
        cells = []
        for col, fmt in zip(_LATEX_METRIC_COLS, _LATEX_METRIC_FMT):
            cell = fmt.format(row[col])
            if std_df is not None and method_label in std_df.index and pd.notna(std_df.loc[method_label, col]):
                cell = f'{cell} $\\pm$ {fmt.format(std_df.loc[method_label, col])}'
            if method_label == best_row[col]:
                cell = f'\\textbf{{{cell}}}'
            cells.append(cell)
        lines.append(f'    {method_name} & {hp} & {" & ".join(cells)} \\\\')
    lines += ['    \\bottomrule', '    \\end{tabular}', '    }']
    return '\n'.join(lines)


def build_denoising_latex_table(metrics_by_folder, folders, caption, label, include_average=True):
    used_folders = [f for f in folders if f[0] in metrics_by_folder]
    panels = [
        _latex_panel(metrics_by_folder[folder_name], string.ascii_lowercase[i], _chip_label(folder_name))
        for i, (folder_name, _) in enumerate(used_folders)
    ]
    if include_average and used_folders:
        avg_df, std_df = _average_metrics_df(metrics_by_folder, used_folders,
                                              metric_cols=_LATEX_METRIC_COLS, return_std=True)
        panels.append(_latex_panel(avg_df, string.ascii_lowercase[len(used_folders)],
                                   f'Mean of All {len(used_folders)} Chips', std_df=std_df))
    body = '\n\n    \\vspace{2.5em}\n\n'.join(panels)
    return (
        '\\begin{table}[htbp]\n'
        '    \\centering\n'
        f'    \\caption{{{caption}}}\n'
        f'    \\label{{{label}}}\n'
        '    \\small\n\n'
        f'{body}\n'
        '\\end{table}'
    )


denoising_latex = build_denoising_latex_table(
    denoising_metrics_by_folder, folders,
    caption='Denoising method comparison across the evaluated chips.',
    label='tab:denoising_comparison',
)
print(denoising_latex)

\begin{table}[htbp]
    \centering
    \caption{Denoising method comparison across the evaluated chips.}
    \label{tab:denoising_comparison}
    \small

    (a) Performance on Chip 01\\[0.5em]
    \resizebox{\textwidth}{!}{
    \begin{tabular}{llrrr}
    \toprule
    \textbf{Method} & \textbf{Hyperparameter} & \textbf{Residual Autocorrelation (Lag-1)} & \textbf{SNR (dB)} & \textbf{Fidelity (Corr)} \\
    \midrule
    Smoothed: Simple Moving Average & (w=50) & 0.199 & 13.32 & 0.963 \\
    SG: Savitzky-Golay & (p=2 w=101) & 0.190 & 13.41 & 0.963 \\
    SG: Savitzky-Golay & (p=3 w=101) & 0.186 & 13.44 & 0.963 \\
    SG: Savitzky-Golay & (p=4 w=101) & \textbf{0.139} & 13.72 & \textbf{0.965} \\
    Wavelet: DWT & (sym4) & 0.201 & 13.37 & 0.962 \\
    Wavelet: DWT & (sym6) & 0.199 & 13.38 & 0.962 \\
    Wavelet: DWT & (sym8) & 0.142 & \textbf{13.73} & 0.965 \\
    \bottomrule
    \end{tabular}
    }

    \vspace{2.5em}

    (b) Performance on Chip 02\\[0.5em]
    \resizebox{\textwidth}{!}{


In [11]:
mnames = ['Residual AC lag-1', 'SNR (dB)', 'Fidelity (corr)']
mascs = [True, False, False]
top_n = 5
for mname, masc in zip(mnames, mascs):
    print(f'\nTop {top_n} methods by {mname}:')
    display(denoising_metrics.sort_values(by=mname, ascending=masc)[[mname]].head(top_n))


Top 5 methods by Residual AC lag-1:


,Residual AC lag-1
Wavelet (sym8),0.122109
SG p=4 (w=113),0.129937
Smoothed (w=50),0.174306
SG p=3 (w=113),0.174693
SG p=2 (w=113),0.179434



Top 5 methods by SNR (dB):


,SNR (dB)
Wavelet (sym8),11.273397
SG p=4 (w=113),11.199587
SG p=3 (w=113),10.915325
Wavelet (sym6),10.911719
SG p=2 (w=113),10.880686



Top 5 methods by Fidelity (corr):


,Fidelity (corr)
Wavelet (sym8),0.943826
SG p=4 (w=113),0.943133
Smoothed (w=50),0.939943
SG p=3 (w=113),0.939775
Wavelet (sym6),0.939568
